# 02 — Split design and leakage controls

This notebook documents the frozen MonReader evaluation populations. Split construction lives in `src/splits.py`; this notebook reads the committed registrations, validates the frozen manifests, and explains which generalization question each split answers.


In [ ]:
from pathlib import Path
import csv
import json
import sys
import yaml

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run from the repository root or notebooks/ directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.splits import validate_dataset_split_compatibility, validate_split

SPLIT_DIR = ROOT / "manifests" / "splits"


## Reviewed video → environment/session map

The conservative minimum grouping unit is the canonical video. A stricter environment/session map was reviewed from early/middle/late frames for every video. Videos were merged only where repeated book/document identity, layout, camera/table geometry, hands, lighting, and numbering context supported the relationship. Ambiguous relations remain separate by default.


In [ ]:
environment_review = yaml.safe_load((ROOT / "manifests" / "environment_groups.yaml").read_text(encoding="utf-8"))
video_map = list(csv.DictReader((ROOT / "manifests" / "video_environment_map.csv").open(newline="", encoding="utf-8")))
print(f"reviewed videos: {len(video_map)}")
print(f"environment/session groups: {len(environment_review['groups'])}")
for group in environment_review["groups"]:
    print(f"{group['environment_id']}: {group['description']} — {group['members']}")


## Frozen split registrations

The five public-facing regimes are **Original**, **Random Stratified**, **Source-Safe**, **Temporal-Ordered**, and **Source-Safe + Temporal**. A separate **Video-Disjoint** registration is retained so the effect of broader environment/session grouping can be measured rather than hidden.


In [ ]:
split_summary = json.loads((SPLIT_DIR / "split_summary.json").read_text(encoding="utf-8"))
for split_id, registration in split_summary["splits"].items():
    print(split_id, registration["name"])
    print("  partitions:", registration["summary"]["partition_counts"])
    print("  labels:", registration["summary"]["partition_label_counts"])


## Leakage and chronology checks

Each frozen CSV is revalidated below. Video-disjoint regimes may not place one canonical video in multiple partitions. Source-Safe also requires environment/session disjointness. Temporal-Ordered requires ascending `FrameNumber` with earliest 80% / middle 10% / latest 10% inside each video with at least 20 frames. Shorter videos are excluded from the primary temporal population. Source-Safe + Temporal uses one complete environment/session as the outer test holdout and chronological 90/10 fitting/validation inside the remaining eligible videos.


In [ ]:
for split_id, registration in split_summary["splits"].items():
    path = ROOT / registration["manifest"]["path"]
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    validate_split(rows, split_name=registration["name"])
    print(f"PASS {registration['name']}: {len(rows):,} rows")


## Source-safe hierarchy

`Video-Disjoint` is the minimum source-safe construction. `Source-Safe` is stricter: the four reviewed environment/session groups are kept disjoint. The final test environment is selected deterministically for class balance, with validation selected the same way from the remaining groups. Because only four broader groups are defensible, their partition sizes are necessarily coarser than an 80/10/10 frame split.


In [ ]:
source_safe = split_summary["splits"]["SPLIT-004"]
print(yaml.safe_dump(source_safe["rule"], sort_keys=False))


## Temporal population

The primary temporal analysis excludes videos with fewer than 20 frames rather than pooling them into the main result. The exclusion remains explicit in the split manifest so the evaluation population is auditable.


In [ ]:
temporal = split_summary["splits"]["SPLIT-005"]
combined = split_summary["splits"]["SPLIT-006"]
print("Temporal-Ordered:", temporal["summary"]["partition_counts"])
print("Source-Safe + Temporal:", combined["summary"]["partition_counts"])


## Dataset / split compatibility

The command-line entrypoint validates dataset/split compatibility before execution. This prevents a later experiment from silently pairing a registered split with the wrong dataset variant.


In [ ]:
compatibility = ROOT / "manifests" / "dataset_split_compatibility.yaml"
for split_id in split_summary["splits"]:
    validate_dataset_split_compatibility(compatibility, dataset_id="DATA-001", split_id=split_id)
print("PASS: all frozen Phase-3 splits are compatible with the canonical original-RGB dataset")


## Phase-3 conclusion

The evaluation populations are frozen before model development expands. The supplied split remains a benchmark reference, frame-random performance is separated from source-safe performance, broader grouping is explicit and reviewable, chronology is enforced centrally, and exact decoded-pixel duplicate leakage is checked against the canonical dataset hashes.
